# Module 2 Exercise: Language Doesn't Matter

Same query, two languages, same plan.

We create a view on top of `customers`, inspect its plan in **Spark SQL**, then
repeat the same filter via the **DataFrame API**. Both end up with the same
Catalyst optimised plan because the language is just a front-end.


## 1. Create a temporary view on `customers`

In [ ]:
%%sql
CREATE OR REPLACE TEMPORARY VIEW german_customers AS
SELECT *
FROM   customers
WHERE  country = 'Germany'


## 2. EXPLAIN via Spark SQL

`EXPLAIN` is the Spark SQL keyword for a query plan. Same idea as `SET SHOWPLAN_XML ON`
in SQL Server.

In [ ]:
%%sql
EXPLAIN
SELECT * FROM german_customers LIMIT 10


## 3. Same filter via the DataFrame API

No SQL this time. Notice that the first operators in the plan are identical to
what Spark SQL produced above.

In [ ]:
df = spark.table("german_customers")
df = df.filter(df["city"] == "Berlin")
df.explain()


## 4. Try it on `orders`

Swap the source table and confirm the same pattern holds on 50M rows.
Count is wrapped in an `EXPLAIN` so you see the plan rather than a number.

In [ ]:
%%sql
EXPLAIN
SELECT COUNT(*)
FROM   orders
WHERE  product = 'Laptop'
  AND  order_date >= date_sub(current_date(), 30)


## Key takeaway

Catalyst reads Spark SQL and DataFrame code, turns both into the same logical plan,
applies the same optimisation rules, and hands the same physical plan to the
executors. **Pick the language you find easier to read; the engine doesn't care.**
